In [244]:
import pandas as pd
import numpy as np

In [246]:
# get 50 states + DC for all datasets
locations = pd.read_csv('../data/location.csv').iloc[:52]
# convert population to be in units of 10,000
locations["population"] = locations["population"]/1e4
# make dataframe holding population of each state in columns
pop_data = locations.loc[:,["location","population"]].T
pop_data.columns = pop_data.iloc[0].str.lstrip('0')
pop_data = pop_data[1:].reset_index(drop=True)

## Normalize and Format Covid-19/Flu datasets

In [249]:
def format_normalize_dataset(df, population_df, ):
    df = df[locations.location.values]
    # Remove leading zeros, for consistency with other merges
    df.columns = df.columns.str.lstrip('0')
    df = df.drop('US', axis=1)
    # rearrange columns to be the same for both datasets, divide column-wise
    population_df = population_df[df.columns]
    return df.divide(population_df.iloc[0], axis=1)
    

In [251]:
# Covid-19 Confirmed Cases per 10k ppl
cases = pd.read_csv('../data/covid19_incident_cases.csv',parse_dates=['date']).set_index('date')
normalized_cases = format_normalize_dataset(cases, pop_data)
normalized_cases.to_csv('../data/covid19_incident_cases_normalized.csv')

In [253]:
# Covid-19 Hospitalizations per 10k ppl
hosp = pd.read_csv('../data/covid19_incident_hosp.csv',parse_dates=['date']).set_index('date')
normalized_hosp = format_normalize_dataset(hosp, pop_data)
normalized_hosp.to_csv('../data/covid19_incident_hosp_normalized.csv')

In [255]:
# Influenza Hospitalizations per 10k ppl
flu_hosp = pd.read_csv('../data/flu_incident_hosp.csv',parse_dates=['date']).set_index('date')
normalized_flu_hosp = format_normalize_dataset(flu_hosp, pop_data)
normalized_flu_hosp.to_csv('../data/flu_incident_hosp_normalized.csv')

## Get State Population Centers

In [258]:
# Data from 2020 US Census
src = "https://www2.census.gov/geo/docs/reference/cenpop2020/CenPop2020_Mean_ST.txt"
centers = pd.read_csv(src)
centers["STATEFP"] = centers["STATEFP"].astype(str)
fips_df = pd.DataFrame(centers["STATEFP"])
centers_df = (fips_df.merge(fips_df, how="cross") # cartesian product of state FIPS codes
              # get latitude, longitude of origin state
                     .merge(centers.loc[:,["STATEFP","LATITUDE", "LONGITUDE"]],
                            left_on="STATEFP_x", right_on="STATEFP", how="left")
                     .drop(columns=["STATEFP"])
              # get latitude, longitude of destination state
                     .merge(centers.loc[:,["STATEFP","LATITUDE", "LONGITUDE"]],
                            left_on="STATEFP_y", right_on="STATEFP", how="left")
                     .drop(columns=["STATEFP"])
              # make column names more descriptive
                     .rename(columns={
                         "STATEFP_x": "origin",
                         "STATEFP_y": "destination",
                         "LATITUDE_x": "origin_lat",
                         "LONGITUDE_x": "origin_long",
                         "LATITUDE_y": "destination_lat",
                         "LONGITUDE_y": "destination_long"
                         }))

### Calculate distance between state population centers

In [272]:
def haversine_distance(lat1, long1, lat2, long2,decimals=2):
    """
    Distance between two pairs of latitude, longitude coordinates according to haversine formula.
    """
    # convert all coords to radians
    lat1, long1, lat2, long2 = map(np.radians, [lat1, long1, lat2, long2])
    # earth's radius (km)
    r =  6378.137
    # inside sqrt part of haversine formula
    inside_sqrt = np.sin((lat2-lat1)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin((long2-long1)/2)**2
    return  round(2*r*np.arcsin(np.sqrt(inside_sqrt)),decimals) 

In [274]:
centers_df['distance'] = haversine_distance(centers_df['origin_lat'],
                                           centers_df['origin_long'],
                                           centers_df['destination_lat'],
                                           centers_df['destination_long']
                                           )
# remove self loops
centers_df = centers_df.loc[centers_df.origin != centers_df.destination,:]
centers_formatted = centers_df.loc[:,["origin","destination","distance"]]
centers_formatted.to_csv('../data/state_populationcenters_distance.csv', index=False)

## Process Patchflow Datasets
Reformat unnormalized flow datasets for total (bidirectional) flow between states and organize into 3 column format needed for SGPE to make spatial embeddings

In [266]:
# Patchflow uses GADM codes not FIPS codes, need to convert
# Dict from https://gadm.org/maps/USA_1.html
GADM_codes = {'GADM': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 
                       17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 
                       31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 
                       45, 46, 47, 48, 49, 50, 51], 
              'location_name': ['Maine', 'Washington', 'North Dakota', 'Montana', 'Vermont', 
                       'New Hampshire', 'Minnesota', 'Massachusetts', 'Oregon', 
                       'Michigan', 'New York', 'Rhode Island', 'Idaho', 'Wisconsin', 
                       'Connecticut', 'South Dakota', 'Wyoming', 'New Jersey', 
                       'Pennsylvania', 'Iowa', 'Ohio', 'Delaware', 'Maryland', 
                       'Nebraska', 'District of Columbia', 'Indiana', 'Illinois', 
                       'Nevada', 'West Virginia', 'Utah', 'Virginia', 'Colorado', 
                       'California', 'Missouri', 'Kentucky', 'Kansas', 'North Carolina', 
                       'Tennessee', 'South Carolina', 'Oklahoma', 'Arkansas', 'Arizona', 
                       'New Mexico', 'Georgia', 'Alabama', 'Mississippi', 'Texas', 
                       'Louisiana', 'Florida', 'Alaska', 'Hawaii']
}
# make datafrmae containing all states and their respective GADM and FIPS codes
gadm_df = pd.DataFrame(GADM_codes)
converter_df = (centers.loc[:,["STATEFP","STNAME"]].
                rename(columns={
                    "STATEFP":"FIPS",
                    "STNAME":"location_name"}).
                # merge dataset of statenames and fips codes with GADM dataset
                merge(gadm_df, on="location_name", how="left")
               )
# convert GADM codes to str and fill na values with 0 (puerto rico in FIPS data not in GADM data)
converter_df["GADM"] = converter_df["GADM"].fillna(0)
converter_df["GADM"] = converter_df["GADM"].astype(int).astype(str)
# GADM to FIPS lookup table
GADM_to_FIPS = converter_df.loc[:,["GADM","FIPS"]].set_index("GADM").to_dict()["FIPS"]

In [268]:
# Patch flow data has flow with direction, use total birdirectional flow for undirected metric
def get_total_flow(flow_df):
    """
    Gets the total flow between two locations, according to radiation model, by 
    summing the flow to and from a location. Not normalized by population
    """
    # Get the GADM code without extra characters
    flow_df['origin'] = (flow_df['origin'].str.replace('USA.', '').
                         str.replace('_1', ''))
    flow_df['destination'] = (flow_df['destination'].str.replace('USA.', '').
                         str.replace('_1', ''))
    # remove self loops
    flow_df = flow_df.loc[flow_df.origin != flow_df.destination,:]
    # If original data has flow from A -> B, reverse flow is B -> A
    flow_reversed = flow_df.rename(columns={'origin': 'destination', 
                                            'destination': 'origin', 
                                            'flow': 'reverse_flow'})
    flow_merged = pd.merge(flow_df, flow_reversed, on=['origin', 'destination'], 
                           how='outer')
    # Fill NAs with 0 for no flow
    flow_merged['flow'] = flow_merged['flow'].fillna(0)
    flow_merged['reverse_flow'] = flow_merged['reverse_flow'].fillna(0)
    # Get bi-directional flow
    flow_merged['total_flow'] = flow_merged['flow'] + flow_merged['reverse_flow']
    # Convert to FIPS, drop directional flow columns
    final_df = flow_merged.drop(['flow','reverse_flow'], axis=1)
    final_df['origin'] = final_df['origin'].map(GADM_to_FIPS) 
    final_df['destination'] = final_df['destination'].map(GADM_to_FIPS) 
    return final_df

In [270]:
# patchflow admin level 1 data cloned from:
# https://github.com/NSSAC/patchflow-data/tree/main/data/v1.0/USA
rad_constants = [0.01, 0.02, 0.05, 0.1, 0.2, 0.5]
for r in rad_constants:
    src = "../data/patchflow_raw/USA_admin1_radiation_constant_{:.02f}.csv".format(r)
    patch_df = pd.read_csv(src).drop("time", axis=1)
    flow_data = get_total_flow(patch_df)
    path = "../data/flows_processed/USA_admin1_radiation_constant_{:.02f}_processed.csv".format(r)
    flow_data.to_csv(path, index=False)